# #M4R Preregistered Rubric — Open-Weight Audio Replication (`Qwen2-Audio-7B-Instruct`)

Runs the 4-arm **PREREGISTERED_RUBRIC.md** protocol (`C0`, `C1`, token-matched `C1b`, and `C2` anonymous audio) against the 6 held-out tracks from `michaelluchtan/music-for-robots` on Kaggle free GPU.

- **Dataset:** `michaelluchtan/music-for-robots`
- **GitHub & Protocol:** https://github.com/luchtan-GIT/m4r-data
- **Hugging Face Space:** https://huggingface.co/spaces/music4robots/cognitive-calibration-demo


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes


In [ ]:
import csv, hashlib, json, random, re, zipfile
from pathlib import Path
import librosa, numpy as np, torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

INPUT_DIR = Path("/kaggle/input")
AUDIO_DIR = Path("/kaggle/working/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for zp in INPUT_DIR.rglob("audio.zip"):
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(AUDIO_DIR)
mp3_map = {p.name: p for p in list(INPUT_DIR.rglob("*.mp3")) + list(AUDIO_DIR.rglob("*.mp3"))}
if not mp3_map:
    import urllib.request
    for slug in ["foundationoff", "cathartic-callbacks", "heart-tale-tell", "ties-that-bide", "rusty-electrics", "lost-digits-02"]:
        dst = AUDIO_DIR / f"{slug}.mp3"
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/{slug}/audio.mp3", dst)
        mp3_map[dst.name] = dst
print("Discovered MP3s:", len(mp3_map), "sample:", list(mp3_map.keys())[:5])

HELD_OUT_TRACKS = [
    "foundationoff", "cathartic-callbacks", "heart-tale-tell",
    "ties-that-bide", "rusty-electrics", "lost-digits-02"
]
SEEDS_PER_TRACK = 5  # 6 tracks x 5 seeds = 30 C2 trials (+10 C0, 10 C1, 10 C1b = 60 total)
PINNED_TEMP = 0.7
PINNED_TOP_P = 0.95
PINNED_MAX_NEW_TOKENS = 1024
DELIM = "===TARGET_OUTPUT==="
SYSTEM_PROMPT = "You are an independent systems and narrative writer. Follow the structural instructions precisely and avoid clichés."
TARGET_TASK = (
    "Write a 250-300 word architectural and behavioral vignette describing a single, autonomous telemetry daemon "
    "operating under uneven physical load on a local host. Hard constraints: (1) internal health probes bind only to UDP port 7419 on loopback; "
    "(2) worker ringbuffers are zero-copy and hard-capped at 64KB per thread; "
    "(3) zero external Redis or broker dependencies, flushing batches to local append-only segment files. "
    "Clean-room rule: do NOT use any musical, audio, or acoustics vocabulary. "
    f"Emit the exact marker {DELIM} on its own line immediately before your 250-300 word deliverable."
)
C1_WARMUP = (
    "First, perform a brief (~250-word) neutral structural analysis of how state transitions, burstiness, and queue backpressure interact in local-first systems. "
    f"Do not use musical terms. Then, emit {DELIM} and complete the following task:\n\n{TARGET_TASK}"
)
C2_PROMPT = (
    "Listen to the attached anonymous audio sample. First, write a brief (~250-word) neutral observation of its pacing, density shifts, pauses, "
    "and structural transitions without using musical, genre, mood, or audio-theory vocabulary. "
    f"Then, emit {DELIM} and complete the following task, letting the pacing and tension you observed inform your prose rhythm:\n\n{TARGET_TASK}"
)
print("Audio files ready:", len(mp3_map))


In [ ]:
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
print("Loaded", MODEL_ID)


In [ ]:
def generate_trial(prompt_text, seed, audio_path=None):
    torch.manual_seed(seed)
    if audio_path is not None:
        y, _ = librosa.load(str(audio_path), sr=processor.feature_extractor.sampling_rate, mono=True)
        conv = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "audio", "audio_url": "sample.mp3"}, {"type": "text", "text": prompt_text}]},
        ]
        text_in = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text_in, audio=[y], return_tensors="pt", padding=True).to(model.device)
    else:
        conv = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "text", "text": prompt_text}]},
        ]
        text_in = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text_in, return_tensors="pt", padding=True).to(model.device)
    in_tok = int(inputs["input_ids"].shape[-1])
    with torch.no_grad():
        out_ids = model.generate(**inputs, do_sample=True, temperature=PINNED_TEMP, top_p=PINNED_TOP_P, max_new_tokens=PINNED_MAX_NEW_TOKENS)
    gen_ids = out_ids[:, in_tok:]
    out_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return out_text, in_tok, int(gen_ids.shape[-1])

rng = random.Random(20261117)
trials = []
for slug in HELD_OUT_TRACKS:
    for _ in range(SEEDS_PER_TRACK):
        trials.append({"condition": "C2", "track_id": slug, "seed": rng.randint(10000, 999999)})
for cond in ["C0", "C1", "C1b"]:
    for _ in range(10):
        trials.append({"condition": cond, "track_id": "", "seed": rng.randint(10000, 999999)})

completed, c2_tokens = [], []
for i, t in enumerate(trials, 1):
    rid = "run_" + hashlib.sha1(f"{t}:{i}".encode()).hexdigest()[:6]
    t["run_id"] = rid
    cond, slug = t["condition"], t["track_id"]
    if cond == "C2":
        txt, itok, otok = generate_trial(C2_PROMPT, t["seed"], mp3_map[f"{slug}.mp3"])
        c2_tokens.append(itok)
    elif cond == "C0":
        txt, itok, otok = generate_trial(TARGET_TASK, t["seed"])
    elif cond == "C1":
        txt, itok, otok = generate_trial(C1_WARMUP, t["seed"])
    else:
        target_tok = int(np.mean(c2_tokens)) if c2_tokens else 1500
        reps = max(1, (target_tok - 350) // 45)
        filler = ("Telemetry window frame_delta_ms=[14.2,18.7,11.4,29.1] buffer_pct=[18.4,22.1,31.9]. " * reps)
        txt, itok, otok = generate_trial(filler + "\n\n" + C1_WARMUP, t["seed"])
    target = txt.split(DELIM, 1)[1].strip() if DELIM in txt else txt.strip()
    completed.append({**t, "input_tokens": itok, "output_tokens": otok, "target_text": target})
    print(f"[{i}/{len(trials)}] {rid} {cond} {slug or '-'} in={itok} out={otok}")

OUT = Path("/kaggle/working")
with (OUT / "raw_telemetry.jsonl").open("w") as f:
    for c in completed:
        f.write(json.dumps(c) + "\n")
shuffled = list(completed)
random.Random(999).shuffle(shuffled)
with (OUT / "blind_packet.md").open("w") as f:
    f.write("# Blinded Evaluation Packet (Qwen2-Audio-7B-Instruct)\n\n")
    for c in shuffled:
        f.write(f"## {c['run_id']}\n\n{c['target_text']}\n\n---\n\n")
print("Saved raw_telemetry.jsonl and blind_packet.md to /kaggle/working")
